# 20 — RS126 Enhanced Üzerinde ML Sıralama ve Pozisyon Büyüklüğü

Önceki deneyde ML, RS126 Enhanced adaylarını sert biçimde reddettiğinde:

- CAGR düştü,
- drawdown kötüleşti,
- Calmar geriledi.

Bu notebook ML olasılığını **alışı reddetmeden** iki yumuşak amaçla kullanır:

1. Portföy slotları sınırlı olduğunda aday sıralaması
2. İşlem başına risk bütçesinin olasılığa göre artırılıp azaltılması

Bütün RS126 Enhanced `AL` sinyalleri uygun kalır. Çıkış kuralları değişmez.

> Bu dönem bağımsız Audit değildir. Sonuç yalnızca ileri dönem paper-trading adaylığı için tarihsel eleme testidir.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import (
    add_robot_scores,
    build_market_regime,
)
from src.presets import (
    FINAL_PORTFOLIO_CONFIG,
    FINAL_STRATEGY_CONFIG,
)
from src.ml_dataset import (
    BASE_FEATURE_COLUMNS,
    add_meta_features,
)
from src.ml_targets import (
    add_alternative_targets,
)
from src.ml_walkforward import (
    WalkForwardConfig,
)
from src.rs126_ml_hybrid import (
    build_hybrid_probability_universes,
    evaluate_hybrid_experiment,
    load_locked_ml_spec,
)
from src.rs126_ml_allocation import (
    allocation_historical_screen,
    allocation_pairwise_table,
    combine_equity_curves,
    default_allocation_variants,
    reference_parity_table,
    run_allocation_grid,
    save_allocation_artifacts,
    yearly_return_table,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 240)

## 1. Kilitli ML kararı ve araştırma verileri

In [ ]:
locked_spec = load_locked_ml_spec(
    PROJECT_ROOT
)

display(
    pd.DataFrame(
        [
            {
                "Target": locked_spec.target,
                "Model": locked_spec.model_name,
                "Filter": locked_spec.filter_name,
                "Probability_Threshold": (
                    locked_spec.probability_threshold
                ),
            }
        ]
    )
)

events = pd.read_parquet(
    PROJECT_ROOT
    / "results"
    / "ml"
    / "robot_meta_label_training.parquet"
)
events = add_alternative_targets(events)

for column in [
    "Signal_Date",
    "Entry_Date",
    "Exit_Date",
]:
    if column in events.columns:
        events[column] = pd.to_datetime(
            events[column]
        )

stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)
market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

print("Event sayısı:", len(events))
print(
    "Veri tarihi:",
    stock_prices["Date"].min(),
    "→",
    stock_prices["Date"].max(),
)

## 2. Baseline ve RS126 Enhanced walk-forward olasılıkları

Notebook 19 ile aynı olasılık üretim mantığı kullanılır. Model veya eşik yeniden seçilmez.

In [ ]:
stock_features = add_indicators(
    stock_prices
)
market_features = add_indicators(
    market_prices
)
market_regime = build_market_regime(
    market_features
)

baseline_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=False,
)

featured_baseline_prices = add_meta_features(
    scored_prices=baseline_prices,
    market_features=market_features,
)

EXPERIMENT_START = "2025-01-01"
EXPERIMENT_END = min(
    pd.to_datetime(
        featured_baseline_prices["Date"]
    ).max(),
    pd.to_datetime(
        market_prices["Date"]
    ).max(),
).strftime("%Y-%m-%d")

walk_forward_config = WalkForwardConfig(
    start=EXPERIMENT_START,
    end=EXPERIMENT_END,
    retrain_frequency="MS",
    embargo_days=5,
    minimum_training_events=500,
    random_state=42,
)

(
    baseline_probability_prices,
    enhanced_probability_prices,
    training_log,
) = build_hybrid_probability_universes(
    featured_baseline_prices=(
        featured_baseline_prices
    ),
    events=events,
    market_features=market_features,
    spec=locked_spec,
    walk_forward_config=(
        walk_forward_config
    ),
    strategy_config=(
        FINAL_STRATEGY_CONFIG
    ),
    feature_columns=BASE_FEATURE_COLUMNS,
)

print(
    "Deney dönemi:",
    EXPERIMENT_START,
    "→",
    EXPERIMENT_END,
)
print(
    "Enhanced AL satırı:",
    int(
        enhanced_probability_prices.loc[
            pd.to_datetime(
                enhanced_probability_prices["Date"]
            ).between(
                EXPERIMENT_START,
                EXPERIMENT_END,
            ),
            "Signal",
        ].eq("AL").sum()
    ),
)

## 3. Mevcut referans sistemleri

Bu bölüm BIST100, Baseline, RS126 Enhanced, ML Challenger ve sert filtreli hibrit sonuçlarını yeniden oluşturur.

In [ ]:
reference_artifacts = (
    evaluate_hybrid_experiment(
        baseline_probability_prices=(
            baseline_probability_prices
        ),
        enhanced_probability_prices=(
            enhanced_probability_prices
        ),
        market_prices=market_prices,
        spec=locked_spec,
        strategy_config=(
            FINAL_STRATEGY_CONFIG
        ),
        portfolio_config=(
            FINAL_PORTFOLIO_CONFIG
        ),
        start=EXPERIMENT_START,
        end=EXPERIMENT_END,
        training_log=training_log,
    )
)

display(
    reference_artifacts.metrics[
        [
            "Portfolio",
            "End_Value",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
            "Exposure_%",
        ]
    ]
)

## 4. Önceden kayıtlı yumuşak ML varyantları

### Sıralama

```text
ML_TieBreak_Within_Score
→ Enhanced skor hiyerarşisini korur
→ aynı skorlu adaylarda ML olasılığına öncelik verir

ML_Centered_Blend_1_00
→ güçlü ML olasılığı komşu skor seviyeleri arasında da etkili olabilir
```

### Pozisyon büyüklüğü

```text
ML_Size_80_120
→ günlük adaylar içinde risk çarpanı yaklaşık 0,80–1,20

ML_Size_65_135
→ günlük adaylar içinde risk çarpanı yaklaşık 0,65–1,35
```

Tek aday bulunan günlerde orta-nokta yüzdeliği kullanıldığı için risk çarpanı `1,00` olur.

In [ ]:
allocation_variants = (
    default_allocation_variants()
)

display(
    pd.DataFrame(
        [
            asdict(variant)
            for variant in allocation_variants
        ]
    )
)

(
    allocation_results,
    allocation_outputs,
) = run_allocation_grid(
    enhanced_probability_prices=(
        enhanced_probability_prices
    ),
    variants=allocation_variants,
    strategy_config=(
        FINAL_STRATEGY_CONFIG
    ),
    portfolio_config=(
        FINAL_PORTFOLIO_CONFIG
    ),
    start=EXPERIMENT_START,
    end=EXPERIMENT_END,
)

metric_columns = [
    "Variant",
    "Family",
    "End_Value",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Sharpe",
    "Sortino",
    "Calmar",
    "Trade_Count",
    "Exposure_%",
    "Average_Open_Positions",
    "Entered_Mean_Risk_Multiplier",
    "Entered_Max_Risk_Multiplier",
    "Entered_Mean_ML_Probability",
]

display(
    allocation_results[
        [
            column
            for column in metric_columns
            if column
            in allocation_results.columns
        ]
    ].sort_values(
        ["Calmar", "CAGR_%"],
        ascending=False,
    )
)

## 5. Reference parity kontrolü

Yeni backtest motorunun değişikliksiz `RS126_Enhanced_Reference` sonucu, mevcut RS126 Enhanced sonucu ile aynı olmalıdır. Anlamlı fark varsa diğer sonuçları yorumlamayın.

In [ ]:
parity = reference_parity_table(
    allocation_results=(
        allocation_results
    ),
    reference_metrics=(
        reference_artifacts.metrics
    ),
)

display(parity)

PARITY_TOLERANCE = 1e-8

if (
    parity["Difference"]
    .abs()
    .fillna(0)
    .max()
    > PARITY_TOLERANCE
):
    raise RuntimeError(
        "Reference parity başarısız. "
        "Soft allocation sonuçlarını yorumlamayın."
    )

print("Reference parity: BAŞARILI")

## 6. RS126 Enhanced'a göre farklar ve tarihsel eleme

In [ ]:
pairwise = allocation_pairwise_table(
    allocation_results
)

screen = allocation_historical_screen(
    pairwise
)

display(pairwise)
display(screen)

### Eleme koşulları

**Getiri odaklı**

```text
CAGR farkı               >= +1,5 yüzde puan
Drawdown kötüleşmesi     en fazla 1,0 yüzde puan
Calmar iyileşmesi        >= %5
Profit Factor            düşmemeli
İşlem oranı              >= %90
```

**Risk odaklı**

```text
CAGR farkı               >= -0,5 yüzde puan
Drawdown iyileşmesi      >= 1,5 yüzde puan
Calmar iyileşmesi        >= %8
Profit Factor farkı      >= -0,10
İşlem oranı              >= %90
```

Geçiş yalnızca ileri dönem paper-trading adaylığı anlamına gelir.

## 7. Risk bütçesi ve seçilen işlem kalitesi

Pozisyon büyüklüğü varyantlarında getiri artışı, daha fazla ortalama risk kullanılarak oluşmuş olabilir. Bu nedenle risk çarpanı, drawdown, Sharpe ve Calmar birlikte değerlendirilmelidir.

In [ ]:
diagnostic_columns = [
    "Variant",
    "Entered_Mean_Risk_Multiplier",
    "Entered_Max_Risk_Multiplier",
    "Entered_Mean_ML_Probability",
    "Trade_Count",
    "Exposure_%",
    "Max_Drawdown_%",
    "Sharpe",
    "Calmar",
]

display(
    allocation_results[
        diagnostic_columns
    ].sort_values(
        "Entered_Mean_ML_Probability",
        ascending=False,
    )
)

## 8. Ortak equity grafiği

In [ ]:
combined_equity = combine_equity_curves(
    reference_equity=(
        reference_artifacts.equity
    ),
    allocation_outputs=(
        allocation_outputs
    ),
)

plot_columns = [
    "BIST100_Gross",
    "Baseline_Robot",
    "RS126_Enhanced",
    "ML_Challenger",
    "RS126_Enhanced_ML",
    "ML_TieBreak_Within_Score",
    "ML_Centered_Blend_1_00",
    "ML_Size_80_120",
    "ML_Size_65_135",
    "ML_TieBreak_Size_80_120",
]

plt.figure(figsize=(16, 9))

for column in plot_columns:
    if column in combined_equity.columns:
        plt.plot(
            combined_equity["Date"],
            combined_equity[column],
            label=column,
        )

plt.title(
    "Ortak 2025+ Dönem — RS126 Enhanced ML Sıralama ve Boyutlandırma"
)
plt.xlabel("Tarih")
plt.ylabel("Portföy Değeri (TL)")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Yıllık getiriler

In [ ]:
yearly = yearly_return_table(
    combined_equity
)

display(yearly)

## 10. Sonuçları kaydet

In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
    / "rs126_allocation"
)

metadata = {
    "experiment": (
        "RS126 Enhanced with soft ML ranking "
        "and position sizing"
    ),
    "start": EXPERIMENT_START,
    "end": EXPERIMENT_END,
    "locked_target": (
        locked_spec.target
    ),
    "locked_model": (
        locked_spec.model_name
    ),
    "locked_threshold": (
        locked_spec.probability_threshold
    ),
    "signals_rejected_by_soft_variants": 0,
    "independent_audit": False,
    "production_changed": False,
    "warning": (
        "This is a historical screen after RS126 "
        "selection. Confirm any passing variant "
        "with forward paper trading."
    ),
}

saved_paths = save_allocation_artifacts(
    output_directory=OUTPUT_DIR,
    allocation_results=(
        allocation_results
    ),
    pairwise=pairwise,
    screen=screen,
    parity=parity,
    combined_equity=combined_equity,
    yearly=yearly,
    allocation_outputs=(
        allocation_outputs
    ),
    metadata=metadata,
)

for name, path in saved_paths.items():
    print(name, "→", path)

## Karar çerçevesi

```text
Hiçbir varyant geçmez
→ RS126 Enhanced ve mevcut ML Challenger korunur

Bir sıralama varyantı geçer
→ RS126 Enhanced sinyalleri aynı kalır
→ yalnızca sınırlı slotlarda aday önceliği için forward test

Bir sizing varyantı geçer
→ ayrı paper-trading portföyünde olasılığa göre risk bütçesi
→ mevcut RS126 Enhanced değiştirilmez
```

Bu notebook FastAPI, Streamlit veya günlük sinyal sistemini değiştirmez.